In [119]:
#Библиотеки
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    average_precision_score,
    precision_recall_curve,
    ConfusionMatrixDisplay)
from sklearn.metrics import confusion_matrix
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

In [120]:
#Загрузим датасет
df = pd.read_csv("heart.csv")

#"Чистим" названия столбцов
df.columns = df.columns.str.strip()

#Выведем размер датасета, кол-во дубликатов строк и строк с пропусками
print(f'Размер таблицы до подготовки данных:{df.shape}')
print(f'Количество дубликатов до подготовки = {df.duplicated().sum()}')
print(f'Количество строк с пропусками до подготовки = {df.isna().any(axis=1).sum()}') 

#Проверим фичи - выведем названия, кол-во и % пропусков, типы данных
info = pd.DataFrame({"missing":(df.isna().sum()),
                     "missing_%": (df.isna().mean() * 100).round(2),
                     "dtype": df.dtypes})
print(info.sort_values("missing_%", ascending=False))

# выведем уникальные значения категориального таргета
print("\nУникальные таргеты:")
print(df["HeartDisease"].unique().tolist())

#смотрим распределение таргета
print(df["HeartDisease"].value_counts()) 


df.head(5)

Размер таблицы до подготовки данных:(918, 12)
Количество дубликатов до подготовки = 0
Количество строк с пропусками до подготовки = 0
                missing  missing_%    dtype
Age                   0        0.0    int64
Sex                   0        0.0      str
ChestPainType         0        0.0      str
RestingBP             0        0.0    int64
Cholesterol           0        0.0    int64
FastingBS             0        0.0    int64
RestingECG            0        0.0      str
MaxHR                 0        0.0    int64
ExerciseAngina        0        0.0      str
Oldpeak               0        0.0  float64
ST_Slope              0        0.0      str
HeartDisease          0        0.0    int64

Уникальные таргеты:
[0, 1]
HeartDisease
1    508
0    410
Name: count, dtype: int64


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [121]:
# выведем уникальные значения категориальных переменных
categorical_features = ["Sex", "ChestPainType", "FastingBS", "RestingECG", "ExerciseAngina", "ST_Slope"]

for feature in categorical_features:
    print(f"Уникальные {feature}: {df[feature].unique().tolist()}")

Уникальные Sex: ['M', 'F']
Уникальные ChestPainType: ['ATA', 'NAP', 'ASY', 'TA']
Уникальные FastingBS: [0, 1]
Уникальные RestingECG: ['Normal', 'ST', 'LVH']
Уникальные ExerciseAngina: ['N', 'Y']
Уникальные ST_Slope: ['Up', 'Flat', 'Down']


Таким образом, видим:
- размер датасета (918, 12)
- дубликаты и пропуски отсутствуют
- таргет категориальный (1/0), фичи: 5 типа строка (str), 6 числовых (int, float). НО категориальных по смыслу - 6 (FastingBS уже закодирован в бинарный)
- распределение таргета ориентировочно 5к4

In [122]:
df.describe() #смотрим статистику числовых фичей

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease
count,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000
mean,53.510893,132.396514,198.799564,0.233115,136.809368,0.887364,0.553377
std,9.432617,18.514154,109.384145,0.423046,25.460334,1.066570,0.497414
min,28.000000,0.000000,0.000000,0.000000,60.000000,-2.600000,0.000000
25%,47.000000,120.000000,173.250000,0.000000,120.000000,0.000000,0.000000
50%,54.000000,130.000000,223.000000,0.000000,138.000000,0.600000,1.000000
75%,60.000000,140.000000,267.000000,0.000000,156.000000,1.500000,1.000000
max,77.000000,200.000000,603.000000,1.000000,202.000000,6.200000,1.000000


Видим, что у фичей RestingBP и Cholesterol - минимальные значения равны 0. Такого не может быть, в связи с чем предлагаю добавить индикатор пропуска и заменить медианой. 

Соберем препроцессор через ColumnTransformer с 3 видами обработки:
- заполнение пропусков (0) медианой с индикацией пропуска для RestingBP и Cholesterol
- масштабирование через числовых фичей через StandardScaler для Age, RestingBP, Cholesterol, MaxHR, Oldpeak
- OHE для категориальных столбцов Sex, ChestPainType, RestingECG, ExerciseAngina, ST_Slope

In [123]:
categorical_features = [
    "Sex",
    "ChestPainType",
    "RestingECG",
    "ExerciseAngina",
    "ST_Slope"]

zero_missing_features = ["RestingBP", "Cholesterol"]

scale_features = ["Age", "MaxHR", "Oldpeak"]

missing_pipeline = Pipeline([
    ("imputer", SimpleImputer(
            missing_values=0,
            strategy="median",
            add_indicator=True)),
    ("scaler", StandardScaler())])

preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("zero_missing", missing_pipeline, zero_missing_features),
    ("numeric", StandardScaler(), scale_features)])

In [124]:
X = df.drop(columns=["HeartDisease"])     
y = df["HeartDisease"]     

#Делим данные на train (80%) и test (20%) с stratify по целевой переменной
RANDOM_STATE = 42 #фиксируем случайность
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

Обучим базовую модель KNN c параметрами по умолчанию, используя кросс валидацию, и посчитаем метрики Accuracy, F1 и ROC‑AUC.

In [125]:
from sklearn.neighbors import KNeighborsClassifier

# Базовый KNN с параметрами по умолчанию
knn_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier())])

# Стратифицированная кросс-валидация
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42)

# Метрики
scoring = {
    "Accuracy": "accuracy",
    "F1": "f1",
    "ROC-AUC": "roc_auc"}

# CV только на train
cv_knn = cross_validate(
    knn_model,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1)

# Достаём метрики Train и Validation
accuracy_train = cv_knn["train_Accuracy"]
accuracy_val = cv_knn["test_Accuracy"]

f1_train = cv_knn["train_F1"]
f1_val = cv_knn["test_F1"]

roc_auc_train = cv_knn["train_ROC-AUC"]
roc_auc_val = cv_knn["test_ROC-AUC"]

# Создаём таблицу
metrics_knn = pd.DataFrame({
    "Accuracy Train": [
        accuracy_train.mean().round(3),
        accuracy_train.std().round(3),
        accuracy_train.round(3),
        f"{accuracy_train.min():.3f} - {accuracy_train.max():.3f}"],

    "Accuracy Val": [
        accuracy_val.mean().round(3),
        accuracy_val.std().round(3),
        accuracy_val.round(3),
        f"{accuracy_val.min():.3f} - {accuracy_val.max():.3f}"],

    "F1 Train": [
        f1_train.mean().round(3),
        f1_train.std().round(3),
        f1_train.round(3),
        f"{f1_train.min():.3f} - {f1_train.max():.3f}"],

    "F1 Val": [
        f1_val.mean().round(3),
        f1_val.std().round(3),
        f1_val.round(3),
        f"{f1_val.min():.3f} - {f1_val.max():.3f}"],

    "ROC-AUC Train": [
        roc_auc_train.mean().round(3),
        roc_auc_train.std().round(3),
        roc_auc_train.round(3),
        f"{roc_auc_train.min():.3f} - {roc_auc_train.max():.3f}"],

    "ROC-AUC Val": [
        roc_auc_val.mean().round(3),
        roc_auc_val.std().round(3),
        roc_auc_val.round(3),
        f"{roc_auc_val.min():.3f} - {roc_auc_val.max():.3f}"]},
index=[
    "Среднее",
    "Стандартное отклонение",
    "Значения по фолдам",
    "Разброс"])

metrics_knn

,Accuracy Train,Accuracy Val,F1 Train,F1 Val,ROC-AUC Train,ROC-AUC Val
Среднее,0.883,0.854,0.897,0.87,0.958,0.902
Стандартное отклонение,0.009,0.037,0.008,0.033,0.007,0.035
Значения по фолдам,"[0.877, 0.901, 0.877, 0.882, 0.876]","[0.844, 0.789, 0.878, 0.898, 0.863]","[0.891, 0.912, 0.893, 0.897, 0.891]","[0.861, 0.812, 0.89, 0.909, 0.88]","[0.961, 0.97, 0.953, 0.948, 0.958]","[0.887, 0.844, 0.926, 0.946, 0.908]"
Разброс,0.876 - 0.901,0.789 - 0.898,0.891 - 0.912,0.812 - 0.909,0.948 - 0.970,0.844 - 0.946


Выводы: 
- Явных признаков сильного переобучения нет, т.к. разрыв между Train–Validation естественен, хотя по ROC-AUC он уже немного заметнее.
- Недообучения тоже не видно, т.к. validation-метрики достаточно высокие, особенно ROC-AUC.
- Стандартные отклонения на validation небольшие, поэтому результат достаточно стабилен между фолдами. Правда, отдельные фолды всё-таки отличаются.

Подбор гиперпараметров KNN: GridSearchCV

In [126]:
from sklearn.model_selection import GridSearchCV
import time

# Метрики
scoring = {
    "Accuracy": "accuracy",
    "F1": "f1",
    "ROC-AUC": "roc_auc"}

# Сетка гиперпараметров
param_grid_knn = {
    "classifier__n_neighbors": range(1, 31),
    "classifier__weights": ["uniform", "distance"],
    "classifier__metric": ["minkowski"],
    "classifier__p": [1, 2]}

grid_knn = GridSearchCV(
    estimator=knn_model,
    param_grid=param_grid_knn,
    cv=skf,
    scoring=scoring,
    refit="Accuracy",
    return_train_score=True,
    n_jobs=-1)

# Засекаем время
start_time = time.perf_counter()
grid_knn.fit(X_train, y_train)
grid_time = time.perf_counter() - start_time

# Таблица лучших гиперпараметров
best_params_comparison = pd.DataFrame({
    "n_neighbors": [grid_knn.best_params_["classifier__n_neighbors"]],
    "weights": [grid_knn.best_params_["classifier__weights"]],
    "metric": [grid_knn.best_params_["classifier__metric"]],
    "p": [grid_knn.best_params_["classifier__p"]]}, index=["GridSearchCV"])

display(best_params_comparison)

# Результаты GridSearchCV
results_grid = grid_knn.cv_results_
best_idx_grid = grid_knn.best_index_

# Таблица результатов
search_comparison = pd.DataFrame({
    "GridSearchCV": [
        round(results_grid["mean_test_Accuracy"][best_idx_grid], 3),
        round(results_grid["mean_test_F1"][best_idx_grid], 3),
        round(results_grid["mean_test_ROC-AUC"][best_idx_grid], 3),
        round(grid_time, 3)]},
index=[
    "Accuracy",
    "F1",
    "ROC-AUC",
    "Время, сек."])

search_comparison

,n_neighbors,weights,metric,p
GridSearchCV,5,uniform,minkowski,1


,GridSearchCV
Accuracy,0.868
F1,0.884
ROC-AUC,0.911
"Время, сек.",4.019


Подбор гиперпараметров KNN: RandomizedSearchCV

In [127]:
from sklearn.model_selection import RandomizedSearchCV

# Сетка гиперпараметров
param_dist_knn = {
    "classifier__n_neighbors": range(1, 31),
    "classifier__weights": ["uniform", "distance"],
    "classifier__metric": ["minkowski"],
    "classifier__p": [1, 2]}

# RandomizedSearchCV
random_knn = RandomizedSearchCV(
    estimator=knn_model,
    param_distributions=param_dist_knn,
    n_iter=30,  #Попробуем 30 итераций, что в 4 раза меньше, чем при GridSearchCV (120)
    cv=skf,
    scoring=scoring,
    refit="Accuracy",
    return_train_score=True,
    random_state=42,
    n_jobs=-1)

# Засекаем время
start_time = time.perf_counter()
random_knn.fit(X_train, y_train)
random_time = time.perf_counter() - start_time

# Добавляем лучшие параметры в таблицу
best_params_comparison.loc["RandomizedSearchCV"] = [
    random_knn.best_params_["classifier__n_neighbors"],
    random_knn.best_params_["classifier__weights"],
    random_knn.best_params_["classifier__metric"],
    random_knn.best_params_["classifier__p"]]

display(best_params_comparison)

results_random = random_knn.cv_results_
best_idx_random = random_knn.best_index_

search_comparison["RandomizedSearchCV"] = [
    round(random_knn.best_score_, 3),
    round(results_random["mean_test_F1"][best_idx_random], 3),
    round(results_random["mean_test_ROC-AUC"][best_idx_random], 3),
    round(random_time, 3)]

search_comparison

,n_neighbors,weights,metric,p
GridSearchCV,5,uniform,minkowski,1
RandomizedSearchCV,12,distance,minkowski,1


,GridSearchCV,RandomizedSearchCV
Accuracy,0.868,0.862
F1,0.884,0.880
ROC-AUC,0.911,0.920
"Время, сек.",4.019,1.040


В качестве эксперимента запустим RandomizedSearchCV на 120 итерациях

In [128]:
# Сетка гиперпараметров
param_dist_knn = {
    "classifier__n_neighbors": range(1, 31),
    "classifier__weights": ["uniform", "distance"],
    "classifier__metric": ["minkowski"],
    "classifier__p": [1, 2]}

# RandomizedSearchCV
random_knn = RandomizedSearchCV(
    estimator=knn_model,
    param_distributions=param_dist_knn,
    n_iter=120,  #Попробуем 30 итераций, что в 4 раза меньше, чем при GridSearchCV (120)
    cv=skf,
    scoring=scoring,
    refit="Accuracy",
    return_train_score=True,
    random_state=42,
    n_jobs=-1)

# Засекаем время
start_time = time.perf_counter()
random_knn.fit(X_train, y_train)
random_time = time.perf_counter() - start_time

# Добавляем лучшие параметры в таблицу
best_params_comparison.loc["RandomizedSearchCV_max"] = [
    random_knn.best_params_["classifier__n_neighbors"],
    random_knn.best_params_["classifier__weights"],
    random_knn.best_params_["classifier__metric"],
    random_knn.best_params_["classifier__p"]]

display(best_params_comparison)

results_random = random_knn.cv_results_
best_idx_random = random_knn.best_index_

search_comparison["RandomizedSearchCV_max"] = [
    round(random_knn.best_score_, 3),
    round(results_random["mean_test_F1"][best_idx_random], 3),
    round(results_random["mean_test_ROC-AUC"][best_idx_random], 3),
    round(random_time, 3)]

search_comparison

,n_neighbors,weights,metric,p
GridSearchCV,5,uniform,minkowski,1
RandomizedSearchCV,12,distance,minkowski,1
RandomizedSearchCV_max,5,uniform,minkowski,1


,GridSearchCV,RandomizedSearchCV,RandomizedSearchCV_max
Accuracy,0.868,0.862,0.868
F1,0.884,0.880,0.884
ROC-AUC,0.911,0.920,0.911
"Время, сек.",4.019,1.040,4.108


Экспериментально выяснили, что RandomizedSearchCV с количеством итераций, равным полной сетке гиперпараметров в GridSearchCV, выдает такой же результат, как и GridSearchCV (оптимальные гиперпараметры, а соответственно и значения метрик)

Теперь сравним подбор гиперпараметров модели KNN методами GridSearchCV и RandomizedSearchCV:

| Метод | Число комбинаций | Время, сек. | Accuracy | F1 | ROC-AUC | n_neighbors | weights | metric | p |
|---|---:|---:|---:|---:|---:|---:|---|---|---:|
| GridSearchCV | 120 | 3.925 | **0.868** | **0.884** | 0.911 | 5 | uniform | minkowski | 1 |
| RandomizedSearchCV | 30 | **1.035** | 0.862 | 0.880 | **0.920** | 12 | distance | minkowski | 1 |

По результатам сравнения GridSearchCV и RandomizedSearchCV были получены разные оптимальные настройки KNN (за исключением metric=minkowski и p=1. Оба метода выбрали манхэттенское расстояние).

RandomizedSearchCV оказался заметно быстрее (примерно в 3.8 раза) Это объясняется тем, что RandomizedSearchCV проверил только 30 комбинаций гиперпараметров вместо 120.

Существенной разницы в качестве моделей не наблюдается. GridSearchCV показал немного более высокие Accuracy и F1, тогда как RandomizedSearchCV получил более высокий ROC-AUC. Поскольку основной метрикой является Accuracy, GridSearchCV формально показал лучший результат, однако преимущество всего 0.006, поэтому RandomizedSearchCV продемонстрировал сопоставимое качество при значительно меньшем времени поиска.

Сравним с моделью Random Forest

In [129]:
from sklearn.ensemble import RandomForestClassifier

# Препроцессор для Random Forest
preprocessor_rf = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("zero_missing", SimpleImputer(missing_values=0, strategy="median", add_indicator=True), zero_missing_features)
], remainder="passthrough")


# Random Forest
rf_model = Pipeline([
    ("preprocessor", preprocessor_rf),
    ("classifier", RandomForestClassifier(random_state=42))])

# Сетка гиперпараметров
param_grid_rf = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [None, 5, 10, 15],
    "classifier__max_features": ["sqrt", "log2", None],
    "classifier__min_samples_split": [2, 5]}

# Параметры GridSearchCV
grid_rf = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid_rf,
    cv=skf,
    scoring=scoring,
    refit="Accuracy",
    return_train_score=True,
    n_jobs=-1)

grid_rf.fit(X_train, y_train)

print("Лучшие гиперпараметры Random Forest:")
print(grid_rf.best_params_)

print(f"Лучший Validation Accuracy: {grid_rf.best_score_:.3f}")

results_rf = grid_rf.cv_results_
best_idx_rf = grid_rf.best_index_

# Создаем функцию, которая достает метрики фолдов из GridSearchCV
def get_fold_scores(results, best_idx, metric, dataset):
    prefix = "train" if dataset == "train" else "test"

    scores = np.array([
        results[f"split{i}_{prefix}_{metric}"][best_idx]
        for i in range(skf.n_splits)])

    return scores

# Достаем метрики фолдов
acc_train_rf = get_fold_scores(results_rf, best_idx_rf, "Accuracy", "train")
acc_val_rf = get_fold_scores(results_rf, best_idx_rf, "Accuracy", "val")

f1_train_rf = get_fold_scores(results_rf, best_idx_rf, "F1", "train")
f1_val_rf = get_fold_scores(results_rf, best_idx_rf, "F1", "val")

roc_train_rf = get_fold_scores(results_rf, best_idx_rf, "ROC-AUC", "train")
roc_val_rf = get_fold_scores(results_rf, best_idx_rf, "ROC-AUC", "val")

# Формируем таблицу результатов
metrics_rf = pd.DataFrame({
    "Accuracy Train": [
        acc_train_rf.mean().round(3),
        acc_train_rf.std().round(3),
        acc_train_rf.round(3),
        f"{acc_train_rf.min():.3f} - {acc_train_rf.max():.3f}"],

    "Accuracy Val": [
        acc_val_rf.mean().round(3),
        acc_val_rf.std().round(3),
        acc_val_rf.round(3),
        f"{acc_val_rf.min():.3f} - {acc_val_rf.max():.3f}"],

    "F1 Train": [
        f1_train_rf.mean().round(3),
        f1_train_rf.std().round(3),
        f1_train_rf.round(3),
        f"{f1_train_rf.min():.3f} - {f1_train_rf.max():.3f}"],

    "F1 Val": [
        f1_val_rf.mean().round(3),
        f1_val_rf.std().round(3),
        f1_val_rf.round(3),
        f"{f1_val_rf.min():.3f} - {f1_val_rf.max():.3f}"],

    "ROC-AUC Train": [
        roc_train_rf.mean().round(3),
        roc_train_rf.std().round(3),
        roc_train_rf.round(3),
        f"{roc_train_rf.min():.3f} - {roc_train_rf.max():.3f}"],

    "ROC-AUC Val": [
        roc_val_rf.mean().round(3),
        roc_val_rf.std().round(3),
        roc_val_rf.round(3),
        f"{roc_val_rf.min():.3f} - {roc_val_rf.max():.3f}"]},
index=[
    "Среднее",
    "Стандартное отклонение",
    "Значения по фолдам",
    "Разброс"])

display(metrics_rf)
display(metrics_knn)

Лучшие гиперпараметры Random Forest:
{'classifier__max_depth': 10, 'classifier__max_features': 'sqrt', 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100}
Лучший Validation Accuracy: 0.864


,Accuracy Train,Accuracy Val,F1 Train,F1 Val,ROC-AUC Train,ROC-AUC Val
Среднее,0.992,0.864,0.993,0.88,1.0,0.928
Стандартное отклонение,0.002,0.033,0.002,0.026,0.0,0.028
Значения по фолдам,"[0.991, 0.997, 0.991, 0.991, 0.99]","[0.857, 0.803, 0.884, 0.898, 0.877]","[0.992, 0.997, 0.992, 0.992, 0.991]","[0.873, 0.834, 0.892, 0.909, 0.893]","[1.0, 1.0, 1.0, 1.0, 1.0]","[0.922, 0.877, 0.943, 0.954, 0.944]"
Разброс,0.990 - 0.997,0.803 - 0.898,0.991 - 0.997,0.834 - 0.909,1.000 - 1.000,0.877 - 0.954


,Accuracy Train,Accuracy Val,F1 Train,F1 Val,ROC-AUC Train,ROC-AUC Val
Среднее,0.883,0.854,0.897,0.87,0.958,0.902
Стандартное отклонение,0.009,0.037,0.008,0.033,0.007,0.035
Значения по фолдам,"[0.877, 0.901, 0.877, 0.882, 0.876]","[0.844, 0.789, 0.878, 0.898, 0.863]","[0.891, 0.912, 0.893, 0.897, 0.891]","[0.861, 0.812, 0.89, 0.909, 0.88]","[0.961, 0.97, 0.953, 0.948, 0.958]","[0.887, 0.844, 0.926, 0.946, 0.908]"
Разброс,0.876 - 0.901,0.789 - 0.898,0.891 - 0.912,0.812 - 0.909,0.948 - 0.970,0.844 - 0.946


1) У модели Random Forest все метрики (Accuracy, F1, ROC-AUC) на валидационных данных выше, а их разброс меньше. То есть качество и устойчивость выше.  
2) Однако он сильнее склонен к переобучению (наблюдается заметный разрыв между обучающими и валидационными метриками).  
3) Кроме того, деревья легче интерпретировать по сравнению с моделью KNN, однако с ансамблями немного сложнее, поэтому я бы не стал говорить, что в интерпретируемости Случайный лес выигрывает у KNN


Реализуем Байесовскую оптимизацию подбора гиперпараметров для модели KNN

In [130]:
from hyperopt import hp, tpe, fmin, Trials, space_eval
from sklearn.model_selection import cross_val_score

# Создаем словарь пространства поиска гиперпараметров
space = {
    "n_neighbors": hp.quniform("n_neighbors", 1, 30, 1),   # quniform возвращает числовое значение как float, поэтому позже мы делаем int(...)
    "weights": hp.choice("weights", ["uniform", "distance"]),
    "metric": hp.choice("metric", ["minkowski"]),
    "p": hp.choice("p", [1, 2])}

# Создаем функцию, которую Hyperopt будет вызывать с очередной комбинацией гиперпараметров, которую tpe решил проверить
def objective(params):

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", KNeighborsClassifier(
            n_neighbors=int(params["n_neighbors"]),     # переводим в целое (int)
            weights=params["weights"],
            metric=params["metric"],
            p=params["p"]))])

    accuracy = cross_val_score(
        model,
        X_train,
        y_train,
        cv=skf,
        scoring="accuracy",
        n_jobs=-1).mean()      # берем среднее accuracy всех фолдов

    return -accuracy    # потому что fmin() ищет минимальное значение

# Создаём объект, в котором Hyperopt будет хранить всю историю поиска
trials = Trials()

# Засекаем время
start_time = time.perf_counter()

# Засекаем поиск
best_hyperopt = fmin(
    fn=objective,  # качество каждой комбинации оценивай с помощью нашей функции objective
    space=space,   # гиперпараметры бери из нашего пространства
    algo=tpe.suggest, # Используем TPE. После накопления предыдущих результатов он анализирует, какие значения параметров чаще давали хорошие результаты, и на основании этого предлагает следующие
    max_evals=30,   # провести 30 испытаний (30 комбинаций параметров) 
    trials=trials,  # результаты записывать в созданный нами объект trials
    rstate=np.random.default_rng(42))  # Фиксируем генератор случайных чисел

# Считаем время
hyperopt_time = time.perf_counter() - start_time

# переводим внутренние индексы обратно в названия параметров
best_params_hyperopt = space_eval(
    space,
    best_hyperopt)

# число соседей переводим обратно в int
best_params_hyperopt["n_neighbors"] = int(
    best_params_hyperopt["n_neighbors"])

# Выводим лучшие параметры
print("Лучшие гиперпараметры Hyperopt:")
print(best_params_hyperopt)

# Выводим лучшую метрику accuracy из всей истории Trials (Проходим по каждому trial и достаём его loss)
best_accuracy_hyperopt = -min(
    trial["result"]["loss"]
    for trial in trials.trials)

print("Лучший Accuracy:", round(best_accuracy_hyperopt, 3))
print("Время Hyperopt:", round(hyperopt_time, 3), "сек.")

print(f"Байесовская оптимизация выполнила 30 итераций, при этом лучший результат был найден на {np.argmin([trial["result"]["loss"] for trial in trials.trials]) + 1} итерации")

100%|██████████| 30/30 [00:01<00:00, 22.89trial/s, best loss: -0.8664989283384588]
Лучшие гиперпараметры Hyperopt:
{'metric': 'minkowski', 'n_neighbors': 10, 'p': 1, 'weights': 'distance'}
Лучший Accuracy: 0.866
Время Hyperopt: 1.314 сек.
Байесовская оптимизация выполнила 30 итераций, при этом лучший результат был найден на 24 итерации


Добавим результаты в таблицу

In [131]:
best_params_comparison.loc["TPE"] = [
    best_params_hyperopt["n_neighbors"],
    best_params_hyperopt["weights"],
    best_params_hyperopt["metric"],
    best_params_hyperopt["p"]]

display(best_params_comparison)

best_tpe_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier(
        n_neighbors=best_params_hyperopt["n_neighbors"],
        weights=best_params_hyperopt["weights"],
        metric=best_params_hyperopt["metric"],
        p=best_params_hyperopt["p"]))])

tpe_f1 = cross_val_score(
    best_tpe_model,
    X_train,
    y_train,
    cv=skf,
    scoring="f1",
    n_jobs=-1).mean()

tpe_roc_auc = cross_val_score(
    best_tpe_model,
    X_train,
    y_train,
    cv=skf,
    scoring="roc_auc",
    n_jobs=-1).mean()

search_comparison["TPE"] = [
    round(best_accuracy_hyperopt, 3),
    round(tpe_f1, 3),
    round(tpe_roc_auc, 3),
    round(hyperopt_time, 3)]

display(search_comparison)

,n_neighbors,weights,metric,p
GridSearchCV,5,uniform,minkowski,1
RandomizedSearchCV,12,distance,minkowski,1
RandomizedSearchCV_max,5,uniform,minkowski,1
TPE,10,distance,minkowski,1


,GridSearchCV,RandomizedSearchCV,RandomizedSearchCV_max,TPE
Accuracy,0.868,0.862,0.868,0.866
F1,0.884,0.880,0.884,0.883
ROC-AUC,0.911,0.920,0.911,0.919
"Время, сек.",4.019,1.040,4.108,1.314


Теперь ради эксперимента увеличим максимальное количество итераций для TPE до 120

In [132]:
from hyperopt import hp, tpe, fmin, Trials, space_eval
from sklearn.model_selection import cross_val_score

# Создаем словарь пространства поиска гиперпараметров
space = {
    "n_neighbors": hp.quniform("n_neighbors", 1, 30, 1),   # quniform возвращает числовое значение как float, поэтому позже мы делаем int(...)
    "weights": hp.choice("weights", ["uniform", "distance"]),
    "metric": hp.choice("metric", ["minkowski"]),
    "p": hp.choice("p", [1, 2])}

# Создаем функцию, которую Hyperopt будет вызывать с очередной комбинацией гиперпараметров, которую tpe решил проверить
def objective(params):

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", KNeighborsClassifier(
            n_neighbors=int(params["n_neighbors"]),     # переводим в целое (int)
            weights=params["weights"],
            metric=params["metric"],
            p=params["p"]))])

    accuracy = cross_val_score(
        model,
        X_train,
        y_train,
        cv=skf,
        scoring="accuracy",
        n_jobs=-1).mean()      # берем среднее accuracy всех фолдов

    return -accuracy    # потому что fmin() ищет минимальное значение

# Создаём объект, в котором Hyperopt будет хранить всю историю поиска
trials = Trials()

# Засекаем время
start_time = time.perf_counter()

# Засекаем поиск
best_hyperopt = fmin(
    fn=objective,  # качество каждой комбинации оценивай с помощью нашей функции objective
    space=space,   # гиперпараметры бери из нашего пространства
    algo=tpe.suggest, # Используем TPE. После накопления предыдущих результатов он анализирует, какие значения параметров чаще давали хорошие результаты, и на основании этого предлагает следующие
    max_evals=120,   # провести 30 испытаний (30 комбинаций параметров) 
    trials=trials,  # результаты записывать в созданный нами объект trials
    rstate=np.random.default_rng(42))  # Фиксируем генератор случайных чисел

# Считаем время
hyperopt_time = time.perf_counter() - start_time

# переводим внутренние индексы обратно в названия параметров
best_params_hyperopt = space_eval(
    space,
    best_hyperopt)

# число соседей переводим обратно в int
best_params_hyperopt["n_neighbors"] = int(
    best_params_hyperopt["n_neighbors"])

# Выводим лучшие параметры
print("Лучшие гиперпараметры Hyperopt:")
print(best_params_hyperopt)

# Выводим лучшую метрику accuracy из всей истории Trials (Проходим по каждому trial и достаём его loss)
best_accuracy_hyperopt = -min(
    trial["result"]["loss"]
    for trial in trials.trials)

print("Лучший Accuracy:", round(best_accuracy_hyperopt, 3))
print("Время Hyperopt:", round(hyperopt_time, 3), "сек.")

print(f"Байесовская оптимизация выполнила 120 итераций, при этом лучший результат был найден на {np.argmin([trial["result"]["loss"] for trial in trials.trials]) + 1} итерации")

best_params_comparison.loc["TPE_max"] = [
    best_params_hyperopt["n_neighbors"],
    best_params_hyperopt["weights"],
    best_params_hyperopt["metric"],
    best_params_hyperopt["p"]]

display(best_params_comparison)

best_tpe_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier(
        n_neighbors=best_params_hyperopt["n_neighbors"],
        weights=best_params_hyperopt["weights"],
        metric=best_params_hyperopt["metric"],
        p=best_params_hyperopt["p"]))])

tpe_f1 = cross_val_score(
    best_tpe_model,
    X_train,
    y_train,
    cv=skf,
    scoring="f1",
    n_jobs=-1).mean()

tpe_roc_auc = cross_val_score(
    best_tpe_model,
    X_train,
    y_train,
    cv=skf,
    scoring="roc_auc",
    n_jobs=-1).mean()

search_comparison["TPE_max"] = [
    round(best_accuracy_hyperopt, 3),
    round(tpe_f1, 3),
    round(tpe_roc_auc, 3),
    round(hyperopt_time, 3)]

display(search_comparison)

100%|██████████| 120/120 [00:05<00:00, 22.90trial/s, best loss: -0.8678408349641227]
Лучшие гиперпараметры Hyperopt:
{'metric': 'minkowski', 'n_neighbors': 5, 'p': 1, 'weights': 'uniform'}
Лучший Accuracy: 0.868
Время Hyperopt: 5.244 сек.
Байесовская оптимизация выполнила 120 итераций, при этом лучший результат был найден на 93 итерации


,n_neighbors,weights,metric,p
GridSearchCV,5,uniform,minkowski,1
RandomizedSearchCV,12,distance,minkowski,1
RandomizedSearchCV_max,5,uniform,minkowski,1
TPE,10,distance,minkowski,1
TPE_max,5,uniform,minkowski,1


,GridSearchCV,RandomizedSearchCV,RandomizedSearchCV_max,TPE,TPE_max
Accuracy,0.868,0.862,0.868,0.866,0.868
F1,0.884,0.880,0.884,0.883,0.884
ROC-AUC,0.911,0.920,0.911,0.919,0.911
"Время, сек.",4.019,1.040,4.108,1.314,5.244


Сравнение результата Байесовской оптимизации с GridSearchCV и RandomizedSearchCV:
При одинаковом количестве итераций, равным 30, что в 4 раза меньще общего числа комбинаций для GridSearch, RandomizedSearchCV справился немного быстрее, чем TPE, однако подобрал гиперпараметры, Accuracy при которых немного хуже, чем при методе TPE. 

Необходимо отметить, что при TPE наилучшее качество было найдено уже на 24-й итерации.

Преимуществом вероятностного подхода TPE по сравнению с полным перебором GridSearchCV является в сопоставимом качестве при гораздо более меньшем времени. А по сравнению со случайным поиском RandomizedSearchCV преимущество TPE заключается в лучшем качестве оценки за приблизительно аналогичное количество времени.

Выводы:
Предобработка данных, а именно кодирование, масштабирование и заполнение пропусков, разумеется, положительно повлияло на качество модели. Из всех моделей мы научились подбирать гиперпараметры для KNN тремя разными способами: полный перебор всех гиперпараметров GridSearchCV, перебор случайных комбинаций гиперпараметров RandomizedSearchCV и Байесовскую оптимизацию. Каждый из них имеет свои плюсы и свои минусы. Если мы хотим надежно получить самый лучший результат и при этом можем пожертвовать временем, тогда необходимо выбирать поиск по сетке GridSearch. Если нам важно время, но при этом мы хотим перебрать определенное количество случайных вариантов и выбрать лучший из них, с учетом того, что он не гарантированно будет лучшим, то мы можем использовать RandomizedSearchCV, который нам сэкономит время. Если же мы хотим это время провести не случайный перебор, а умный перебор, то необходимо выбирать байесовскую оптимизацию, которая, по моему мнению, является чем-то средним между GridSearch CV и RandomizedSearch CV в плане качества и времени. 

В качестве модели для практического использования я бы выбрал Random Forest, так как он показал на валидационных данных лучшее значение всех метрик, а их разброс меньше, то есть качество и устойчивость выше. 

Выбор гиперпараметров напрямую влияет на переобучение и недообучение. Например, для модели KNN чем меньше количество соседей, тем более вероятность переобучения. Однако большое значение соседей очевидно приведет к недообучению.